# فصل ۱۳: پردازش داده‌ها در LangChain (Data Processing)
## بخش ۱: اصول پردازش داده و آناتومی کلاس Document

مدل‌های زبانی بزرگ (LLM) مانند GPT-4 یا Claude فوق‌العاده باهوش هستند، اما یک محدودیت فیزیکی اساسی دارند: **«عینک نزدیک‌بین»** یا همان **پنجره زمینه (Context Window)**.

اگر بخواهید یک کتابخانه بزرگ شامل صدها فایل PDF، اسناد شرکتی یا مقالات وب را مستقیماً به مدل بدهید، دو مشکل اساسی رخ می‌دهد:
1. **خطای سرریز (Overflow Error):** طول توکن‌های متن از حداکثر ظرفیت مجاز پرامپت فراتر می‌رود.
2. **پدیده فراموشی در میانه (Lost in the Middle):** حتی اگر متن در پنجره جا شود، مدل در انبوه کلمات گم شده و دقت پاسخگویی به شدت افت می‌کند.

**راهکار مهندسی:** ما باید داده‌های بزرگ را پیش‌پردازش کرده و به **«فیش‌های هوشمند، تمیز و ساخت‌یافته»** تبدیل کنیم تا در زمان نیاز، فقط مرتبط‌ترین فیش‌ها در اختیار مدل قرار گیرند.

### ۱. فرآیند ETL در معماری مدرن RAG

در مهندسی سنتی، مفهوم **ETL** به معنای جریان انتقال داده از منابع خام به انباره‌ها بود. در سیستم‌های هوش مصنوعی و بازیابی اطلاعات (RAG)، مراحل ETL به این صورت تعریف می‌شوند:

```
+-----------------------------------------------------------------------------------------+
|                           AI Data Pipeline (Modern ETL for RAG)                         |
+-----------------------------------------------------------------------------------------+
|                                                                                         |
|  [1. Extract / Load]                                                                    |
|  خواندن فایل‌های خام از منابع پراکنده (PDF, Word, Web, SQL) و تبدیل به محیط پایتون.      |
|  --> ابزار در لنگ‌چین: Document Loaders                                                  |
|                                                                                         |
|-----------------------------------------------------------------------------------------|
|                                                                                         |
|  [2. Transform / Split]                                                                 |
|  پاکسازی متن و قطعه‌قطعه کردن اسناد بزرگ به تکه‌های معنادار (Chunks).                    |
|  --> ابزار در لنگ‌چین: Text Splitters                                                    |
|                                                                                         |
|-----------------------------------------------------------------------------------------|
|                                                                                         |
|  [3. Load / Embed & Store]                                                              |
|  تبدیل تکه‌ها به بردار تعبیه (Embedding) و ذخیره‌سازی در وکتور دیتابیس (فصل ۱۲).        |
|  --> ابزار در لنگ‌چین: Vector Stores (مانند ChromaDB)                                   |
|                                                                                         |
+-----------------------------------------------------------------------------------------+
```

> **تمرکز فصل ۱۳:** تسلط کامل بر مراحل **۱ و ۲ (Load & Split)** و آماده‌سازی استاندارد داده‌ها برای سیستم‌های RAG.

### ۲. آناتومی داده در LangChain: شیء `Document`

شاید تصور کنید وقتی فایلی خوانده می‌شود، خروجی یک رشته متنی ساده (`str`) است. اما در LangChain واحد اتمی داده‌ها، کلاسی به نام **`Document`** است.

چرا متن خالی کافی نیست؟
متن بدون هویت خطرناک است! فرض کنید جمله‌ای داریم: *«سود شرکت در سال جاری ۲۰٪ افزایش یافت.»*
- کدام شرکت؟
- مربوط به کدام سال؟
- از چه منبعی (فایل رسمی یا وبلاگ شخصی) استخراج شده؟

شیء `Document` دو بخش کلیدی دارد:
1. **`page_content` (محتوای اصلی):** متن خام (یک رشته متنی `str`).
2. **`metadata` (شناسنامه):** یک دیکشنری پایتون شامل فراداده‌ها (نام فایل، شماره صفحه، تاریخ، نویسنده، برچسب‌ها، درجه اعتبار و...).

> 💡 **اصل اساسی:** در سیستم‌های حرفه‌ای هوش مصنوعی، **متن بدون متادیتا فاقد ارزش است!**

In [1]:
from langchain_core.documents import Document

# متن خامی که از یک ایمیل اداری به دست آمده است
raw_text = "جلسه مدیران ارشد شرکت به روز دوشنبه ساعت ۱۰ صبح موکول شد."

# ساختن یک شیء Document استاندارد
doc = Document(
    page_content=raw_text,
    metadata={
        "source": "email_inbox",
        "sender": "admin@company.com",
        "date": "2024-11-20",
        "priority": "high",
        "department": "Management"
    }
)

print("--- محتوای اصلی (page_content) ---")
print(doc.page_content)

print("\n--- شناسنامه و فراداده (metadata) ---")
print(doc.metadata)

--- محتوای اصلی (page_content) ---
جلسه مدیران ارشد شرکت به روز دوشنبه ساعت ۱۰ صبح موکول شد.

--- شناسنامه و فراداده (metadata) ---
{'source': 'email_inbox', 'sender': 'admin@company.com', 'date': '2024-11-20', 'priority': 'high', 'department': 'Management'}


### ۳. شبیه‌سازی سناریوی واقعی: حل تناقض اطلاعات با کمک Metadata

فرض کنید می‌خواهید یک ربات پاسخ‌گو به وضعیت بازار بسازید و دو خبر در پایگاه داده شما وجود دارد:
- **خبر ۱ (خبرگزاری معتبر، سال ۲۰۲۴):** «قیمت طلا کاهش یافت.»
- **خبر ۲ (وبلاگ شخصی، سال ۲۰۱۹):** «قیمت طلا به شدت افزایش خواهد یافت.»

اگر این داده‌ها را به صورت رشته‌های ساده (`list[str]`) ذخیره کنید:
```python
database = [
    "قیمت طلا کاهش یافت.",
    "قیمت طلا به شدت افزایش خواهد یافت."
]
```
وقتی کاربر بپرسد: *«وضعیت طلا چطور است؟»*، مدل زبانی با دو گزاره کاملاً متناقض و بدون تاریخ مواجه شده و دچار سردرگمی و توهم می‌شود!

اما با ساختار استاندارد `Document` در LangChain، مسئله به زیبایی حل می‌شود:

In [2]:
news_docs = [
    Document(
        page_content="قیمت طلا کاهش یافت.",
        metadata={
            "source": "isna.ir",
            "year": 2024,
            "trust_score": 10,
            "category": "economy"
        }
    ),
    Document(
        page_content="قیمت طلا به شدت افزایش خواهد یافت.",
        metadata={
            "source": "personal_blog",
            "year": 2019,
            "trust_score": 2,
            "category": "opinion"
        }
    )
]

# نمایش تفاوت اسناد
for i, d in enumerate(news_docs, 1):
    print(f"سند {i}:")
    print(f"  محتوا: {d.page_content}")
    print(f"  منبع:  {d.metadata['source']} (سال {d.metadata['year']}) - نمره اعتبار: {d.metadata['trust_score']}/10\n")

سند 1:
  محتوا: قیمت طلا کاهش یافت.
  منبع:  isna.ir (سال 2024) - نمره اعتبار: 10/10

سند 2:
  محتوا: قیمت طلا به شدت افزایش خواهد یافت.
  منبع:  personal_blog (سال 2019) - نمره اعتبار: 2/10



#### منطق فیلترینگ و تصمیم‌گیری هوشمند:
اکنون خط لوله شما می‌تواند پیش از ارسال اسناد به پرامپت مدل، داده‌های کم‌اعتبار یا قدیمی را فیلتر کند یا اطلاعات منبع را مستقیماً در متن پرامپت بگنجاند:

In [3]:
# فیلتر کردن اسنادی که نمره اعتبار بالای ۵ دارند و در سال جاری (۲۰۲۴) منتشر شده‌اند
valid_docs = [
    d for d in news_docs
    if d.metadata.get("trust_score", 0) >= 5 and d.metadata.get("year") == 2024
]

# آماده‌سازی پرامپت غنی‌شده برای مدل زبانی
context_str = "\n".join([
    f"[منبع: {d.metadata['source']} | سال: {d.metadata['year']}] {d.page_content}"
    for d in valid_docs
])

prompt_to_llm = f"با توجه به اخبار موثق زیر به سوال کاربر پاسخ بده:\n{context_str}\n\nسوال: وضعیت طلا چگونه است؟"
print(prompt_to_llm)

با توجه به اخبار موثق زیر به سوال کاربر پاسخ بده:
[منبع: isna.ir | سال: 2024] قیمت طلا کاهش یافت.

سوال: وضعیت طلا چگونه است؟


### ۴. نقش اشیاء `Document` در ابزارهای بعدی LangChain

تمام ماژول‌هایی که در درس‌های بعدی یاد می‌گیریم، ورودی و خروجی یکسانی بر پایه همین کلاس `Document` دارند:

1. **لودرها (Document Loaders):**
   - وقتی فایلی مانند `PyPDFLoader("book.pdf")` را لود می‌کنید، خروجی لیستی از اشیاء `Document` است (معمولاً هر صفحه کتاب یک `Document` جداگانه با متادیتای `{'page': 1, 'source': 'book.pdf'}`).

2. **اسپلیترها (Text Splitters):**
   - یک `Document` طولانی را دریافت کرده و آن را به ۱۰ شیء `Document` کوچکتر می‌شکنند، در حالی که متادیتای سند والد را به طور خودکار به قطعات فرزند منتقل می‌کنند.

3. **وکتور دیتابیس‌ها (Vector Stores):**
   - متد `vector_db.add_documents(docs)` مستقیماً فیلد `page_content` را بردارسازی کرده و دیکشنری `metadata` را به عنوان فیلدهای فیلترینگ ذخیره می‌کند.

### 🎯 جمع‌بندی نکات کلیدی این درس‌نامه:

1. **محدودیت Context Window:** مدل‌ها نمی‌توانند هزاران صفحه را یکجا بخوانند؛ خطای Overflow و پدیده Lost in the Middle ضرورت پیش‌پردازش را ایجاب می‌کند.
2. **خط لوله ETL در RAG:** استخراج با Document Loaders $\rightarrow$ تبدیل و شکستن با Text Splitters $\rightarrow$ بردارسازی و ذخیره در Vector DB.
3. **کلاس `Document`:** واحد بنیادی داده در لنگ‌چین با دو فیلد الزامی `page_content` (متن) و `metadata` (شناسنامه و اطلاعات جانبی).
4. **ارزش متادیتا:** متن بدون متادیتا در برابر اخبار متناقض و تاریخ‌های متفاوت آسیب‌پذیر است؛ متادیتا کلید فیلترینگ دقیق و جلوگیری از توهم است.